# Notebook 1: Edge Capture Walkthrough

This notebook is the first in a three-part teaching sequence:

1. **Edge capture**: process the bundled audio clip, run Perch, make retention decisions, write an edge database, and create a sender payload.
2. **Hub ingestion**: receive the payload, validate it, store it in the hub database, and run the watchdog.
3. **End-to-end system**: run the whole desktop mock path as a system.

The production script behind this notebook is [`edge_node_mock/src/bio_capture_loop.py`](../edge_node_mock/src/bio_capture_loop.py). We use the same functions, but keep every intermediate object visible.

The repeatable teaching audio is [`notebooks/example_audio/example1_120s_petrel.wav`](example_audio/example1_120s_petrel.wav). It contains wind/sea noise and two grey-faced petrel calls.

This notebook writes reusable artifacts to `notebooks/output/`:

- `edge/edge_notebook.sqlite`: edge database populated from the example clip.
- `edge/retained_audio/*.flac`: retained audio clips.
- `edge/edge_config.notebook.yaml`: generated edge config used by the notebook.
- `transport/payload.msgpack`: sender payload consumed by Notebook 2.
- `transport/payload_summary.json`: small JSON summary of the payload.
- `edge/edge_artifact_manifest.json`: manifest for students to inspect.

## Data Flow

```mermaid
flowchart LR
    A[Example WAV\n120 seconds] --> B[15-second buffers]
    B --> C[3 x 5-second\nPerch windows]
    C --> D[Perch CPU V2]
    D --> E[Logits]
    D --> F[1536-d embeddings]
    E --> G[Noise/Bio/NZ bird scoring]
    G --> H[Retention decision]
    H --> I[Retained FLACs]
    H --> J[Edge SQLite DB]
    F --> J
    J --> K[MessagePack payload]
```

## 1. Setup

Use the project `.venv` as the notebook kernel. The first code cell locates the repository root and imports the same helper functions used by the production scripts.

In [ ]:
from pathlib import Path
import copy
import json
import shutil
import sqlite3
import sys
import time

import msgpack
import numpy as np
import pandas as pd
import soundfile as sf
import yaml
from IPython.display import Audio, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "edge_node_mock").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

pd.set_option("display.max_colwidth", 120)
print("Repository root:", REPO_ROOT)

In [ ]:
from mock_common.config import load_config
from edge_node_mock.src.bio_capture_loop import (
    build_label_index,
    decide_buffer,
    insert_buffer_event,
    iter_audio_buffers,
    load_sqlite_vec,
    run_perch_inference,
    save_retained_audio,
    score_frames,
)
from edge_node_mock.src.init_edge_db import init_edge_db
from edge_node_mock.src.inspect_perch_model import _load_model, load_nz_bird_labels, load_perch_labels, make_perch_windows
from edge_node_mock.src.sender_daemon import build_payload, load_pending_detections, pack_payload

## 2. Prepare A Repeatable Notebook Workspace

The notebooks use `notebooks/output/` as a disposable teaching workspace. The directory is ignored by Git, so students can run and rerun notebooks without creating commit noise.

This first notebook resets the edge and transport outputs because it is the source of the downstream artifacts.

In [ ]:
OUTPUT_ROOT = REPO_ROOT / "notebooks" / "output"
EDGE_OUTPUT = OUTPUT_ROOT / "edge"
TRANSPORT_OUTPUT = OUTPUT_ROOT / "transport"

for path in [EDGE_OUTPUT, TRANSPORT_OUTPUT]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

EXAMPLE_AUDIO_PATH = REPO_ROOT / "notebooks" / "example_audio" / "example1_120s_petrel.wav"
if not EXAMPLE_AUDIO_PATH.exists():
    raise FileNotFoundError(f"Missing example audio: {EXAMPLE_AUDIO_PATH}")

print("Output root:", OUTPUT_ROOT)
print("Example audio:", EXAMPLE_AUDIO_PATH)

## 3. Build A Notebook-Specific Edge Config

The normal edge config points at the full AR4 dataset. For teaching, we copy the example config in memory and change only the paths needed for this notebook.

The key simplification is `include_partial_final_buffer: true`. The teaching clip is just under 120 seconds, so the final 15-second buffer is padded with a tiny amount of silence. That gives students a tidy eight-buffer example.

In [ ]:
BASE_CONFIG_PATH = REPO_ROOT / "edge_node_mock" / "config" / "edge_config.example.yaml"
config = copy.deepcopy(load_config(BASE_CONFIG_PATH))
config.update(
    {
        "raw_audio_mount": str(EXAMPLE_AUDIO_PATH.parent),
        "raw_audio_glob": EXAMPLE_AUDIO_PATH.name,
        "edge_db_path": str(EDGE_OUTPUT / "edge_notebook.sqlite"),
        "retained_audio_dir": str(EDGE_OUTPUT / "retained_audio"),
        "hub_ingest_url": "http://127.0.0.1:8010/ingest_batch",
        "api_key": "notebook-dev-key",
        "include_partial_final_buffer": True,
    }
)

EDGE_CONFIG_PATH = EDGE_OUTPUT / "edge_config.notebook.yaml"
EDGE_CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

for key in ["device_id", "raw_audio_mount", "raw_audio_glob", "edge_db_path", "retained_audio_dir", "include_partial_final_buffer"]:
    print(f"{key}: {config[key]}")

In [ ]:
info = sf.info(EXAMPLE_AUDIO_PATH)
clip_duration = float(info.duration)
expected_buffers = int(np.ceil(clip_duration / 15.0))
print("Sample rate:", info.samplerate)
print("Channels:", info.channels)
print("Duration seconds:", round(clip_duration, 3))
print("Expected 15-second buffers:", expected_buffers)
print("Padding for final buffer seconds:", round(expected_buffers * 15 - clip_duration, 3))

display(Audio(filename=str(EXAMPLE_AUDIO_PATH)))

## 4. Load Labels And Gate Definitions

Perch returns logits by numeric label index. The config names the labels we care about for first-pass gating; the code checks those labels exist and builds numeric lookup tables.

In [ ]:
perch_labels = load_perch_labels(REPO_ROOT / config["perch_label_path"])
nz_labels = load_nz_bird_labels(REPO_ROOT / config["nz_bird_label_path"], perch_labels)
noise_label_indexes = build_label_index(perch_labels, config["noise_labels"], group_name="noise_labels")
bio_label_indexes = build_label_index(perch_labels, config["biological_labels"], group_name="biological_labels")

print("Perch labels:", len(perch_labels))
print("NZ bird subset:", len(nz_labels))
print("Noise label indexes:", noise_label_indexes)
print("Biological label indexes:", bio_label_indexes)

## 5. Stream The Example Audio Into Buffers

This is the mock microphone. Each `AudioBuffer` contains source audio, Perch-ready audio, a timestamp, and the source filename.

In [ ]:
buffers = list(iter_audio_buffers(config, include_partial=True))
rows = []
for buffer in buffers:
    rows.append(
        {
            "buffer_index": buffer.file_buffer_index,
            "timestamp_utc": buffer.timestamp_utc.isoformat(),
            "samples": len(buffer.perch_audio),
            "duration_seconds": len(buffer.perch_audio) / buffer.perch_sample_rate,
            "peak_amplitude": float(np.max(np.abs(buffer.source_audio))),
        }
    )
pd.DataFrame(rows)

In [ ]:
windows = make_perch_windows(buffers[0].perch_audio, buffers[0].perch_sample_rate)
print("Perch windows for one 15-second buffer:", windows.shape)
print("Total 5-second frames in this clip:", len(buffers) * 3)

## 6. Run Perch And Score Every Buffer

The first inference call usually includes TensorFlow warm-up time. That is why we record per-buffer timing as well as the total.

In [ ]:
model_load_start = time.perf_counter()
model, model_source, model_ref = _load_model(config)
model_load_seconds = time.perf_counter() - model_load_start
print("Model source:", model_source)
print("Model reference:", model_ref)
print("Model load seconds:", round(model_load_seconds, 3))

In [ ]:
results = []
dropped_buffer_count = 0
start = time.perf_counter()

for buffer in buffers:
    infer_start = time.perf_counter()
    logits, embeddings = run_perch_inference(model, buffer.perch_audio, buffer.perch_sample_rate)
    infer_seconds = time.perf_counter() - infer_start
    frame_scores = score_frames(
        logits,
        perch_labels=perch_labels,
        noise_label_indexes=noise_label_indexes,
        bio_label_indexes=bio_label_indexes,
        nz_label_indexes=nz_labels,
    )
    decision = decide_buffer(
        frame_scores,
        bio_threshold=float(config["bio_threshold"]),
        noise_threshold=float(config["noise_threshold"]),
        validation_sample_interval=int(config["validation_sample_interval"]),
        dropped_buffer_count=dropped_buffer_count,
    )
    if decision.retention_reason == "dropped":
        dropped_buffer_count += 1
    results.append(
        {
            "buffer": buffer,
            "logits": logits,
            "embeddings": embeddings,
            "frame_scores": frame_scores,
            "decision": decision,
            "infer_seconds": infer_seconds,
        }
    )

total_infer_seconds = time.perf_counter() - start
print("Processed buffers:", len(results))
print("Total inference seconds:", round(total_infer_seconds, 3))

In [ ]:
inference_rows = []
for item in results:
    buffer = item["buffer"]
    decision = item["decision"]
    audio_seconds = len(buffer.perch_audio) / buffer.perch_sample_rate
    inference_rows.append(
        {
            "buffer_index": buffer.file_buffer_index,
            "infer_seconds": item["infer_seconds"],
            "realtime_factor": audio_seconds / item["infer_seconds"],
            "decision": decision.retention_reason,
            "max_bio_label": decision.max_bio_label,
            "max_bio_logit": decision.max_bio_logit,
            "max_perch_label": decision.max_perch_label,
        }
    )

inference_df = pd.DataFrame(inference_rows)
inference_df.style.format({"infer_seconds": "{:.3f}", "realtime_factor": "{:.1f}", "max_bio_logit": "{:.3f}"})

## 7. Inspect The Most Interesting Buffer

We choose the buffer with the highest configured biological score. This is a good moment to connect model output back to what you hear.

In [ ]:
interesting_idx = int(inference_df["max_bio_logit"].astype(float).idxmax())
interesting = results[interesting_idx]
print("Selected buffer:", interesting["buffer"].file_buffer_index)
print("Decision:", interesting["decision"].retention_reason)
print("Max bio:", interesting["decision"].max_bio_label, interesting["decision"].max_bio_logit)
display(Audio(interesting["buffer"].source_audio, rate=interesting["buffer"].source_sample_rate))

In [ ]:
score_rows = []
for score in interesting["frame_scores"]:
    score_rows.append(
        {
            "segment_index": score.segment_index,
            "max_noise_label": score.max_noise_label,
            "max_noise_logit": score.max_noise_logit,
            "max_bio_label": score.max_bio_label,
            "max_bio_logit": score.max_bio_logit,
            "max_perch_label": score.max_perch_label,
            "max_perch_logit": score.max_perch_logit,
        }
    )
pd.DataFrame(score_rows)

In [ ]:
nz_rows = []
for score in interesting["frame_scores"]:
    for rank, bird in enumerate(score.top_nz_birds, start=1):
        nz_rows.append({"segment_index": score.segment_index, "rank": rank, **bird})
pd.DataFrame(nz_rows)

## 8. Save Retained Audio And Write The Edge Database

The production edge node stores every buffer event and every embedding, but only saves full audio for biological hits or validation samples.

This cell writes:

- one `buffer_events` row per 15-second buffer,
- three `embedding_segments` rows per buffer,
- three vector rows per buffer,
- retained `.flac` files for saved buffers.

In [ ]:
edge_db_path = init_edge_db(EDGE_CONFIG_PATH, reset=True)
print("Edge DB:", edge_db_path)

with sqlite3.connect(edge_db_path) as conn:
    conn.execute("PRAGMA foreign_keys=ON;")
    for item in results:
        decision = item["decision"]
        if decision.audio_saved:
            saved_path = save_retained_audio(
                item["buffer"],
                Path(config["retained_audio_dir"]),
                str(config["device_id"]),
                decision.retention_reason,
            )
            decision = type(decision)(**{**decision.__dict__, "filepath": saved_path})
            item["decision"] = decision
        insert_buffer_event(conn, config=config, buffer=item["buffer"], decision=decision, embeddings=item["embeddings"])
    conn.commit()

In [ ]:
with sqlite3.connect(edge_db_path) as conn:
    meta = dict(conn.execute("SELECT key, value FROM schema_metadata;").fetchall())
    if meta["vector_table"] == "perch_vectors":
        load_sqlite_vec(conn)
    counts = {
        "buffer_events": conn.execute("SELECT COUNT(*) FROM buffer_events;").fetchone()[0],
        "embedding_segments": conn.execute("SELECT COUNT(*) FROM embedding_segments;").fetchone()[0],
        meta["vector_table"]: conn.execute(f"SELECT COUNT(*) FROM {meta['vector_table']};").fetchone()[0],
        "pending": conn.execute("SELECT COUNT(*) FROM buffer_events WHERE sync_status='pending';").fetchone()[0],
        "retained_flac": len(list((EDGE_OUTPUT / "retained_audio").glob("*.flac"))),
    }
counts

## 9. Build The MessagePack Sender Payload

Notebook 2 will pretend to be the hub. It will consume the exact MessagePack payload written here.

Notice that embeddings stay as binary data in `payload.msgpack`. That is more compact and faithful to the sender script than converting every vector to JSON.

In [ ]:
detections = load_pending_detections(config)
payload = build_payload(config, detections)
packed = pack_payload(payload)

PAYLOAD_PATH = TRANSPORT_OUTPUT / "payload.msgpack"
PAYLOAD_PATH.write_bytes(packed)

payload_summary = {
    "device_id": payload["device_id"],
    "sent_at_utc": payload["sent_at_utc"],
    "detection_count": len(payload["detections"]),
    "payload_bytes": len(packed),
    "buffer_ids": [d["buffer_id"] for d in payload["detections"]],
    "segments_per_detection": [len(d["embedding_segments"]) for d in payload["detections"]],
    "embedding_bytes_per_segment": len(payload["detections"][0]["embedding_segments"][0]["embedding"]),
    "telemetry": payload["telemetry"],
}
SUMMARY_PATH = TRANSPORT_OUTPUT / "payload_summary.json"
SUMMARY_PATH.write_text(json.dumps(payload_summary, indent=2) + "\n", encoding="utf-8")

payload_summary

## 10. Write A Manifest For The Next Notebook

A manifest is a simple teaching affordance: it tells a new user what files were created and why they matter.

In [ ]:
manifest = {
    "created_by": "01_edge_capture_walkthrough.ipynb",
    "example_audio": str(EXAMPLE_AUDIO_PATH.relative_to(REPO_ROOT)),
    "edge_config": str(EDGE_CONFIG_PATH.relative_to(REPO_ROOT)),
    "edge_db": str(edge_db_path.relative_to(REPO_ROOT)),
    "retained_audio_dir": str((EDGE_OUTPUT / "retained_audio").relative_to(REPO_ROOT)),
    "payload_msgpack": str(PAYLOAD_PATH.relative_to(REPO_ROOT)),
    "payload_summary": str(SUMMARY_PATH.relative_to(REPO_ROOT)),
    "counts": counts,
    "inference": {
        "model_load_seconds": model_load_seconds,
        "total_infer_seconds": total_infer_seconds,
        "mean_infer_seconds": float(inference_df["infer_seconds"].mean()),
    },
}
MANIFEST_PATH = EDGE_OUTPUT / "edge_artifact_manifest.json"
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
manifest

## What To Inspect Before Notebook 2

Open these generated files in your file browser or terminal:

```text
notebooks/output/edge/edge_notebook.sqlite
notebooks/output/edge/retained_audio/
notebooks/output/transport/payload.msgpack
notebooks/output/transport/payload_summary.json
```

Notebook 2 will read `payload.msgpack` and show what the hub does with it.